# Hom-PGD: Polytope-Star Intersection

This tutorial studies a nonconvex feasible set formed by intersecting a convex polytope with a star-shaped domain. Hom-PGD uses a radial map whose gauge is the maximum of the two component gauges.

**Learning goals.** You will: 

1. build the shared polytope-star problem and map;
2. inspect how the component gauges select the active radial boundary;
3. obtain an IPOPT reference for the intersection problem; and
4. visualize a Hom-PGD trajectory in original and latent space.

## 1. Explicit configuration

The constituent polytope is centered at the origin so that it shares the star set's radial center. The problem remains nonconvex because of the star constraint; IPOPT is therefore the explicit numerical reference solver.

In [ ]:
from __future__ import annotations

from pathlib import Path

WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = WORKING_DIR if (WORKING_DIR / 'src' / 'homopt').is_dir() else WORKING_DIR.parents[1]
if not (REPO_ROOT / 'src' / 'homopt').is_dir():
    raise RuntimeError('Launch from the repository root or execute this notebook from tutorials/hom_pgd.')

TUTORIAL_CONFIG = {
    'seed': 2032,
    'device': 'cpu',
    'dtype': 'float32',
    'problem_type': 'poly_star',
    'alpha': 0.30,
    'num_star': 5,
    'poly_config': {
        'n_linear_cons': 6,
        'x_lower': -1.5,
        'x_upper': 1.5,
        'poly_radius': 1.0,
        'angle_offset': 0.20,
    },
    'algorithms': ['Hom-PGD'],
    'max_iterations': 200,
    'max_running_time': 30.0,
    'learning_rate': 3e-2,
    'stepsize_rule': 'adaptive',
    'lr_decay': 0.995,
    'min_lr': 1e-6,
    'common_config': {
        'initial_point_mode': 'random',
        'convergence_threshold': 1e-8,
    },
    'algorithm_config': {
        'Hom-PGD': {'hom_p_norm': 2, 'learning_rate': 3e-2, 'momentum': 0.0},
    },
    'reference_solver': 'ipopt',
    'reference_label': 'IPOPT',
    'reference_ipopt_num_starts': 10,
    'x_grid_bounds': (-1.45, 1.45),
    'x_grid_size': 360,
    'z_grid_size': 220,
    'run_dir': REPO_ROOT / 'results' / 'tutorials' / 'hom_pgd' / '03_poly_star_intersection',
    'figure_dir': REPO_ROOT / 'results' / 'figs' / 'tutorials' / 'hom_pgd' / '03_poly_star_intersection',
}
# Keep each displayed panel at the same physical size in Jupyter and PDF output.
TUTORIAL_FIGURE_STYLE = {
    'panel_width': 5.6,
    'panel_height': 5.0,
    'single_panel': (5.6, 5.0),
    'two_panel': (11.2, 5.0),
}
TUTORIAL_CONFIG


## 2. Build and run the shared benchmark

As in the other tutorials, the benchmark is run exactly once. The returned context is the source for all later calculations and figures.

In [ ]:
import numpy as np

from homopt.experiments.hom_pgd import poly_star_benchmark
from homopt.viz import apply_paper_style_rcparams, build_convex_2d_traces

apply_paper_style_rcparams()
payload, run_context = poly_star_benchmark(
    algorithms=TUTORIAL_CONFIG['algorithms'],
    common_config=TUTORIAL_CONFIG['common_config'],
    algorithm_config=TUTORIAL_CONFIG['algorithm_config'],
    alpha=TUTORIAL_CONFIG['alpha'],
    num_star=TUTORIAL_CONFIG['num_star'],
    problem_type=TUTORIAL_CONFIG['problem_type'],
    poly_config=TUTORIAL_CONFIG['poly_config'],
    max_iterations=TUTORIAL_CONFIG['max_iterations'],
    max_running_time=TUTORIAL_CONFIG['max_running_time'],
    learning_rate=TUTORIAL_CONFIG['learning_rate'],
    stepsize_rule=TUTORIAL_CONFIG['stepsize_rule'],
    lr_decay=TUTORIAL_CONFIG['lr_decay'],
    min_lr=TUTORIAL_CONFIG['min_lr'],
    seed=TUTORIAL_CONFIG['seed'],
    output_dir=TUTORIAL_CONFIG['run_dir'],
    device=TUTORIAL_CONFIG['device'],
    dtype=TUTORIAL_CONFIG['dtype'],
    visualize=False,
    reference_solver=TUTORIAL_CONFIG['reference_solver'],
    reference_label=TUTORIAL_CONFIG['reference_label'],
    reference_ipopt_num_starts=TUTORIAL_CONFIG['reference_ipopt_num_starts'],
    return_context=True,
)
problem = run_context['problem']
hom_map = run_context['hom_map']
reference = run_context['reference']
records = run_context['records']
traces = build_convex_2d_traces(problem, records, TUTORIAL_CONFIG['algorithms'])

{
    'reference_objective': float(reference['objective']),
    'reference_status': reference['status'],
    'run_objective': float(payload['objective']),
    'final_violation': float(run_context['summaries']['Hom-PGD']['final_violation']),
    'recorded_iterates': int(traces['Hom-PGD']['objective'].size),
}


## 3. Active radial boundary by direction

For any nonzero direction `u`, the intersection gauge is

`gamma_intersection(u) = max(gamma_poly(u), gamma_star(u))`.

The larger gauge is the tighter radial constraint. This plot evaluates the two existing component maps and their maximum; it does not construct an alternative approximation.

In [ ]:
import matplotlib.pyplot as plt
import torch

from homopt.viz.artifacts import save_figure
from homopt.viz.style import PAPER_STYLE, apply_paper_axis_style, style_legend

theta = np.linspace(-np.pi, np.pi, 721)
directions = torch.as_tensor(
    np.column_stack([np.cos(theta), np.sin(theta)]),
    dtype=problem.Q.dtype,
    device=problem.device,
)
with torch.no_grad():
    poly_gauge = hom_map.poly_map.gauge(directions, method='autograd').reshape(-1).cpu().numpy()
    star_gauge = hom_map.star_map.gauge(directions).reshape(-1).cpu().numpy()
intersection_gauge = np.maximum(poly_gauge, star_gauge)

fig, ax = plt.subplots(1, 1, figsize=TUTORIAL_FIGURE_STYLE['single_panel'])
ax.plot(theta, poly_gauge, color='#4F79E8', linewidth=PAPER_STYLE['curve_linewidth'], label='Polytope gauge')
ax.plot(theta, star_gauge, color='#C44E52', linewidth=PAPER_STYLE['curve_linewidth'], label='Star gauge')
ax.plot(theta, intersection_gauge, color='#222222', linewidth=PAPER_STYLE['curve_linewidth'], linestyle='--', label='Intersection gauge')
ax.set_xlabel(r'Direction angle $\theta$')
ax.set_ylabel(r'Gauge value $\gamma(u)$')
apply_paper_axis_style(ax, grid=True)
legend = ax.legend(loc='upper center', ncol=1, fontsize=PAPER_STYLE['compact_legend_fontsize'])
style_legend(legend)
gauge_path = save_figure(fig, TUTORIAL_CONFIG['figure_dir'] / 'component_and_intersection_gauges.pdf')
plt.show()
gauge_path


## 4. Intersection geometry and trajectory

The left panel fills only the true intersection. The right panel is the latent unit ball under the combined radial map. The recorded trajectory uses one fixed-seed random latent start and is not reconstructed from a new instance.

In [ ]:
from homopt.viz.landscapes import convex_landscape_grid, convex_z_landscape_grid
from homopt.viz.primitives import draw_convex_landscape, draw_convex_z_landscape, draw_trajectory

low = np.full(2, TUTORIAL_CONFIG['x_grid_bounds'][0], dtype=float)
high = np.full(2, TUTORIAL_CONFIG['x_grid_bounds'][1], dtype=float)
X, Y, objective, inequality, equality, eq_resid = convex_landscape_grid(
    problem, low, high, grid_size=TUTORIAL_CONFIG['x_grid_size']
)
Z1, Z2, z_objective, z_eq_resid, inside, p_norm = convex_z_landscape_grid(
    problem, hom_map, grid_size=TUTORIAL_CONFIG['z_grid_size']
)
trace = traces['Hom-PGD']
x_reference = np.asarray(reference['solution'], dtype=float).reshape(2)

fig, (ax_x, ax_z) = plt.subplots(1, 2, figsize=TUTORIAL_FIGURE_STYLE['two_panel'], constrained_layout=True)
draw_convex_landscape(ax_x, X, Y, objective, inequality, equality, eq_resid)
draw_trajectory(ax_x, trace['trajectory'], label=None)
ax_x.scatter(*x_reference, s=155, marker='*', color='#222222', edgecolor='white', linewidth=0.8, zorder=8)
ax_x.set_title('Polytope-star intersection')

draw_convex_z_landscape(ax_z, Z1, Z2, z_objective, z_eq_resid, inside, p_norm)
draw_trajectory(ax_z, trace['z_trajectory'], label=None)
ax_z.set_title('Latent unit-ball space')

trajectory_path = save_figure(fig, TUTORIAL_CONFIG['figure_dir'] / 'intersection_original_and_latent_trajectory.pdf')
plt.show()
trajectory_path


## Takeaway

The maximum-gauge rule makes the active radial boundary explicit: whichever component gauge is larger in a direction limits the map. Hom-PGD can therefore keep its latent iterates inside a simple unit ball while respecting both parts of the intersection after mapping.